In [45]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db_tugas = client["kopi_nusantara"]
koleksi_menu = db_tugas["menu"]

# Membersihkan data lama agar pengujian mulai dari kondisi bersih
koleksi_menu.delete_many({})

print("Database 'kopi_nusantara' dan collection 'menu' siap.")

Database 'kopi_nusantara' dan collection 'menu' siap.


In [46]:
data_menu = [
    # Kategori Kopi (wajib ada tingkat_kepahitan)
    {"nama_menu": "Espresso Nusantara", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano Gayo", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 4, "stok": 40},
    {"nama_menu": "Kopi Tubruk Toraja", "kategori": "Kopi", "harga": 20000, "tingkat_kepahitan": 5, "stok": 60},
    {"nama_menu": "Kopi Latte Java", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 35},
    {"nama_menu": "Cappuccino Flores", "kategori": "Kopi", "harga": 30000, "tingkat_kepahitan": 3, "stok": 30},
    {"nama_menu": "Kopi Avocado", "kategori": "Kopi", "harga": 35000, "tingkat_kepahitan": 1, "stok": 20},

    # Kategori Makanan (wajib ada pedas_level)
    {"nama_menu": "Nasi Goreng Rempah", "kategori": "Makanan", "harga": 25000, "pedas_level": 2, "stok": 45},
    {"nama_menu": "Mie Goreng Jawa", "kategori": "Makanan", "harga": 22000, "pedas_level": 1, "stok": 50},
    {"nama_menu": "Ayam Geprek Sambal Korek", "kategori": "Makanan", "harga": 28000, "pedas_level": 3, "stok": 25},
    {"nama_menu": "Roti Bakar Cokelat", "kategori": "Makanan", "harga": 15000, "pedas_level": 0, "stok": 40},

    # Kategori Lainnya/Non-Kopi (Atribut Bebas)
    {"nama_menu": "Matcha Latte", "kategori": "Non-Kopi", "harga": 26000, "stok": 30, "rekomendasi_barista": True},
    {"nama_menu": "Lemon Tea", "kategori": "Non-Kopi", "harga": 12000, "stok": 70, "rekomendasi_barista": False}
]

koleksi_menu.insert_many(data_menu)
print("Total dokumen tersimpan di collection 'menu':", koleksi_menu.count_documents({}))

Total dokumen tersimpan di collection 'menu': 12


In [47]:
print("1. Kopi dengan tingkat kepahitan > 3:")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"   - {m['nama_menu']} (Kepahitan: {m['tingkat_kepahitan']})")

1. Kopi dengan tingkat kepahitan > 3:
   - Espresso Nusantara (Kepahitan: 5)
   - Americano Gayo (Kepahitan: 4)
   - Kopi Tubruk Toraja (Kepahitan: 5)


In [48]:
print("\n2. Menu dengan harga 15.000 - 30.000:")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"   - {m['nama_menu']} (Rp{m['harga']:,})")


2. Menu dengan harga 15.000 - 30.000:
   - Espresso Nusantara (Rp18,000)
   - Americano Gayo (Rp22,000)
   - Kopi Tubruk Toraja (Rp20,000)
   - Kopi Latte Java (Rp28,000)
   - Cappuccino Flores (Rp30,000)
   - Nasi Goreng Rempah (Rp25,000)
   - Mie Goreng Jawa (Rp22,000)
   - Ayam Geprek Sambal Korek (Rp28,000)
   - Roti Bakar Cokelat (Rp15,000)
   - Matcha Latte (Rp26,000)


In [49]:
print("\n3. Menu yang tidak memiliki field 'pedas_level':")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"   - {m['nama_menu']} ({m['kategori']})")


3. Menu yang tidak memiliki field 'pedas_level':
   - Espresso Nusantara (Kopi)
   - Americano Gayo (Kopi)
   - Kopi Tubruk Toraja (Kopi)
   - Kopi Latte Java (Kopi)
   - Cappuccino Flores (Kopi)
   - Kopi Avocado (Kopi)
   - Matcha Latte (Non-Kopi)
   - Lemon Tea (Non-Kopi)


In [50]:
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
print("Field 'status' berhasil ditambahkan ke seluruh dokumen.")

Field 'status' berhasil ditambahkan ke seluruh dokumen.


In [51]:
koleksi_menu.update_one({"nama_menu": "Ayam Geprek Sambal Korek"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Kopi Avocado"}, {"$set": {"status": "Habis"}})

print("\nVerifikasi status menu yang diubah:")
for m in koleksi_menu.find({"status": "Habis"}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"   - {m['nama_menu']}: {m['status']}")


Verifikasi status menu yang diubah:
   - Kopi Avocado: Habis
   - Ayam Geprek Sambal Korek: Habis


In [53]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
import pandas as pd
import numpy as np

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas9-MongoDBSpark").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# D.1: Ambil data MongoDB, sesuaikan tipe data & NaN, konversi ke Spark DataFrame
df_pandas = pd.DataFrame(list(koleksi_menu.find({})))
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas = df_pandas.replace({np.nan: None})  # Mencegah TypeError saat ada field kosong (mis. pedas_level / tingkat_kepahitan)

df_spark_menu = spark.createDataFrame(df_pandas)
df_spark_menu.printSchema()
df_spark_menu.show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)



+--------------------+--------------------+--------+-----+-----------------+----+--------+-----------+-------------------+
|                 _id|           nama_menu|kategori|harga|tingkat_kepahitan|stok|  status|pedas_level|rekomendasi_barista|
+--------------------+--------------------+--------+-----+-----------------+----+--------+-----------+-------------------+
|6aebdd495193e16ee...|  Espresso Nusantara|    Kopi|18000|              5.0|  50|Tersedia|       NULL|               NULL|
|6aebdd495193e16ee...|      Americano Gayo|    Kopi|22000|              4.0|  40|Tersedia|       NULL|               NULL|
|6aebdd495193e16ee...|  Kopi Tubruk Toraja|    Kopi|20000|              5.0|  60|Tersedia|       NULL|               NULL|
|6aebdd495193e16ee...|     Kopi Latte Java|    Kopi|28000|              2.0|  35|Tersedia|       NULL|               NULL|
|6aebdd495193e16ee...|   Cappuccino Flores|    Kopi|30000|              3.0|  30|Tersedia|       NULL|               NULL|
|6aebdd495193e16

In [54]:
# D.2: Agregasi rata-rata harga dan jumlah menu per kategori
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()

+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
|    Kopi|        25500.0|          6|
|Non-Kopi|        19000.0|          2|
| Makanan|        22500.0|          4|
+--------+---------------+-----------+



Kesimpulan & Refleksi Skema Data:

Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame terletak pada sifat alami skema dinamis (flexible schema). Dokumen di MongoDB dapat memiliki struktur atribut yang tidak homogen antar-kategori—seperti atribut tingkat_kepahitan yang hanya ada pada menu Kopi dan pedas_level pada Makanan. Ketika data BSON ditarik ke Pandas lalu dipetakan ke Spark DataFrame (yang mengharuskan skema tabel bersifat seragam), kolom-kolom yang tidak dimiliki oleh dokumen tertentu secara otomatis terisi nilai NaN atau null. Untuk mengatasinya, proses transformasi perlu memastikan penyesuaian tipe data (seperti konversi ObjectId ke string) dan penanganan nilai kosong (null handling) dilakukan terlebih dahulu agar Spark dapat membentuk struktur skema terpadu tanpa mengalami type mismatch atau kegagalan parsing.